# Synthetic Data + Capybara Notebook
This notebook prepares each synthetic `.tgl` dataset, extracts frequencies, converts frequencies into Capybara cost vectors, runs Capybara Task 2, and writes 12 Excel sheets (one per generator + regime).

## Import Required Libraries
Import pandas and pathlib, plus Capybara for the Task 2 run.

In [1]:
import sys
from pathlib import Path

# Locate the repository root so the notebook runs from any working directory
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "synthetic_datasets.py").exists())
sys.path.insert(0, str(REPO_ROOT))
import re
from io import StringIO
from pathlib import Path
import pandas as pd
from ete3 import Tree
import math
import time
import numpy as np
import capybara
from synthetic_datasets import list_synthetic_datasets, synthetic_metadata

# Capybara import is attempted later where availability is checked at runtime


## Load and Explore Data
Build an inventory of the `.tgl` synthetic datasets available for processing (their folders are listed in `synthetic_datasets.py`).

In [2]:
# Dataset folders of each generator/regime are listed in synthetic_datasets.py
records = []
for dataset_path, generator, regime in list_synthetic_datasets():
    records.append({
        "dataset_path": dataset_path,
        "generator": generator,
        "regime": regime,
        "file_name": dataset_path.name,
    })

inventory_df = pd.DataFrame(records)
print("Datasets found:", len(inventory_df))
inventory_df.head(20)

Datasets found: 11997


,dataset_path,generator,regime,file_name
0,synthetic_data/asymmetree/high_switch/Datasets...,asymmetree,high_switch,Dataset1.tgl
1,synthetic_data/asymmetree/high_switch/Datasets...,asymmetree,high_switch,Dataset10.tgl
2,synthetic_data/asymmetree/high_switch/Datasets...,asymmetree,high_switch,Dataset100.tgl
3,synthetic_data/asymmetree/high_switch/Datasets...,asymmetree,high_switch,Dataset1000.tgl
4,synthetic_data/asymmetree/high_switch/Datasets...,asymmetree,high_switch,Dataset101.tgl
5,synthetic_data/asymmetree/high_switch/Datasets...,asymmetree,high_switch,Dataset102.tgl
6,synthetic_data/asymmetree/high_switch/Datasets...,asymmetree,high_switch,Dataset103.tgl
7,synthetic_data/asymmetree/high_switch/Datasets...,asymmetree,high_switch,Dataset104.tgl
8,synthetic_data/asymmetree/high_switch/Datasets...,asymmetree,high_switch,Dataset105.tgl
9,synthetic_data/asymmetree/high_switch/Datasets...,asymmetree,high_switch,Dataset106.tgl


## Parse Frequencies and Build Cost Vectors
Extract the frequency vector from each `.tgl` file and transform those frequencies into integer costs for cosp, duplication, switch, and loss.

In [ ]:
frequency_rx = re.compile(r"#Frequencies\s*:\s*\[([^\]]+)\]")
number_rx = r"([0-9.eE+\-]+)"


def parse_event_frequencies(file_path: Path):
    """Return event frequencies as cosp, duplication, switch, and loss.

    Generator conventions:
    - asymmetree stores event counts at the end. Frequencies are count / sum of
      Speciation, Duplication, and Horizontal Gene Transfer; Loss is not included in the denominator.
    - coala stores #Frequencies directly as [cosp, duplication, host-switch, loss].
    - cophylo stores Cospeciation and Host_switch directly as frequencies.
    - treeducken stores Cospeciations and Host_Spreads/switches directly as frequencies.
    """
    text = (REPO_ROOT / file_path).read_text(encoding="utf-8", errors="ignore")
    generator = synthetic_metadata(file_path)[0]

    def _get(pattern):
        matches = re.findall(pattern, text, flags=re.I | re.M)
        if not matches:
            return None
        value = matches[-1]
        if isinstance(value, tuple):
            value = next((v for v in value if v), None)
        try:
            return float(value)
        except (TypeError, ValueError):
            return None

    if generator == "asymmetree":
        speciation = _get(rf"^\s*Speciation\s*:\s*{number_rx}\s*$")
        duplication = _get(rf"^\s*Duplication\s*:\s*{number_rx}\s*$")
        loss = _get(rf"^\s*Loss\s*:\s*{number_rx}\s*$")
        transfer = _get(rf"^\s*Horizontal Gene Transfer\s*:\s*{number_rx}\s*$")
        raw_counts = [speciation, duplication, transfer, loss]
        if any(v is not None for v in raw_counts):
            counts = [0.0 if v is None else v for v in raw_counts]
            total = sum(counts[:3])
            if total > 0:
                speciation, duplication, transfer, loss = counts
                return {
                    "cosp_freq": speciation / total,
                    "dup_freq": duplication / total,
                    "switch_freq": transfer / total,
                    "loss_freq": loss / total if total > 0 else None,
                }

    if generator == "coala":
        match = frequency_rx.search(text)
        if match:
            try:
                values = [float(x.strip()) for x in match.group(1).split(",")]
                if len(values) >= 4:
                    cosp, duplication, switch, loss = values[:4]
                    return {
                        "cosp_freq": cosp,
                        "dup_freq": duplication,
                        "switch_freq": switch,
                        "loss_freq": loss,
                    }
            except ValueError:
                pass

    cosp = None
    switch = None
    if generator == "cophylo":
        cosp = _get(rf"Cospeciation\s*=\s*{number_rx}")
        switch = _get(rf"Host_switch\s*=\s*{number_rx}")
    elif generator == "treeducken":
        cosp = _get(rf"Cospeciations\s+{number_rx}")
        switch = _get(rf"Host_Spreads/switches\s+{number_rx}")
    else:
        cosp = _get(rf"Cospeciations?\s*(?:=|:)??\s+{number_rx}")
        switch = (
            _get(rf"Host_Spreads/switches\s+{number_rx}")
            or _get(rf"Host_switch\s*=\s*{number_rx}")
            or _get(rf"Horizontal Gene Transfer\s*:\s*{number_rx}")
        )

    return {
        "cosp_freq": cosp,
        "dup_freq": None,
        "switch_freq": switch,
        "loss_freq": None,
    }


# Build inventory and attach the event frequencies
freq_df = inventory_df["dataset_path"].apply(parse_event_frequencies).apply(pd.Series)
inventory_df = pd.concat([inventory_df.drop(columns=freq_df.columns.intersection(inventory_df.columns), errors="ignore"), freq_df], axis=1)

# Build task rows: for each dataset produce 4 cost variants (3 classical + 1 transformed)
tasks = []


def enforce_positive_loss_cost(cost_vector):
    cost_vector = list(cost_vector)
    cost_vector[3] = max(1, int(cost_vector[3]))
    return cost_vector


classical_variants = [ [0,1,2,1], [0,1,1,1], [0,1,3,1] ]
for _, row in inventory_df.iterrows():
    base = {
        "dataset_path": row["dataset_path"],
        "generator": row["generator"],
        "regime": row["regime"],
        "file_name": row["file_name"],
        "real_cosp_freq": row.get("cosp_freq", None),
        "cosp_freq": row.get("cosp_freq", None),
        "dup_freq": row.get("dup_freq", None),
        "real_switch_freq": row.get("switch_freq", None),
        "switch_freq": row.get("switch_freq", None),
        "loss_freq": row.get("loss_freq", None),
    }
    # classical variants (do not depend on frequencies)
    for i, cv in enumerate(classical_variants, start=1):
        t = base.copy()
        t.update({"cost_vector": enforce_positive_loss_cost(cv), "cost_label": f"classical_{i}"})
        tasks.append(t)

    # transformed variant: scale -ln(freq) for all available event frequencies before rounding.
    # If duplication or loss frequency is unavailable, use cost 1 for that event.
    transformed_cost_scale = 1000
    c = row.get("cosp_freq")
    d = row.get("dup_freq")
    s = row.get("switch_freq")
    l = row.get("loss_freq")
    def _transform(v, default=None):
        try:
            if v is None or pd.isna(v):
                return default
            fv = float(v)
            if fv <= 0:
                return default
            val = -math.log(fv)
            # Capybara expects integer costs; scaling preserves differences before rounding.
            return int(round(val * transformed_cost_scale))
        except Exception:
            return None

    c_cost = _transform(c)
    d_cost = _transform(d, default=1)
    s_cost = _transform(s)
    l_cost = _transform(l, default=1)

    t = base.copy()
    if c_cost is None or d_cost is None or s_cost is None or l_cost is None:
        t.update({"cost_vector": None, "cost_label": "transformed"})
    else:
        t.update({"cost_vector": [c_cost, d_cost, s_cost, l_cost], "cost_label": "transformed"})
    tasks.append(t)

# tasks_df contains 4 rows per original dataset
tasks_df = pd.DataFrame(tasks)

print("Valid task rows by label:", tasks_df.groupby("cost_label")["cost_vector"].apply(lambda s: s.apply(lambda v: v is not None).sum()).to_dict(), flush=True)

tasks_df.head()


In [ ]:
class SkipDataset(Exception):
    """Raised when a dataset cannot be used after pruning."""


def prune_unmapped_leaves(nexus_text: str) -> str:
    """Prune host/parasite leaves that are not present in the association table.

    This supports multiple .tgl/NEXUS formats produced by different generators:
    - '# HOST_TREE' / '# PARASITE_TREE' / '# ASSOCIATIONS' ... ENDBLOCK
    - 'BEGIN HOST;' / 'BEGIN PARASITE;' / 'BEGIN DISTRIBUTION;' NEXUS blocks
    """
    # try hash-block style first
    host_block = re.search(r"# HOST_TREE\s*(.*?)\s*ENDBLOCK", nexus_text, flags=re.S | re.I)
    parasite_block = re.search(r"# PARASITE_TREE\s*(.*?)\s*ENDBLOCK", nexus_text, flags=re.S | re.I)
    assoc_block = re.search(r"# ASSOCIATIONS\s*(.*?)\s*ENDBLOCK", nexus_text, flags=re.S | re.I)

    # fallback: NEXUS-style BEGIN ... ENDBLOCK/END blocks
    if not host_block:
        host_block = re.search(r"BEGIN HOST;\s*(.*?)\s*ENDBLOCK;", nexus_text, flags=re.S | re.I)
    if not parasite_block:
        parasite_block = re.search(r"BEGIN PARASITE;\s*(.*?)\s*ENDBLOCK;", nexus_text, flags=re.S | re.I)
    if not assoc_block:
        assoc_block = re.search(r"BEGIN DISTRIBUTION;\s*(.*?)\s*(END;|ENDBLOCK;)", nexus_text, flags=re.S | re.I)

    if not (host_block and parasite_block and assoc_block):
        raise ValueError("missing HOST, PARASITE, or DISTRIBUTION block")

    assoc_text = assoc_block.group(1)

    def _parse_associations(text):
        pairs = []
        token_rx = re.compile(r"[^\s:,;]+")
        for line in text.splitlines():
            line = line.strip()
            if not line:
                continue
            line = line.rstrip(",;").strip()
            if not line or line.upper() in {"RANGE", "END", "ENDBLOCK"}:
                continue
            if ":" in line:
                parasite_text, host_text = line.split(":", 1)
                parasite_tokens = token_rx.findall(parasite_text)
                host_tokens = token_rx.findall(host_text)
                if parasite_tokens:
                    pairs.extend((parasite_tokens[0], host) for host in host_tokens)
            else:
                tokens = token_rx.findall(line)
                if len(tokens) >= 2:
                    pairs.append((tokens[0], tokens[1]))
        return pairs

    def _keep_first_host_per_parasite(pairs):
        first_pairs = []
        seen_parasites = set()
        for parasite, host in pairs:
            if parasite in seen_parasites:
                continue
            first_pairs.append((parasite, host))
            seen_parasites.add(parasite)
        return first_pairs

    mapped_pairs = _parse_associations(assoc_text)
    if not mapped_pairs:
        raise SkipDataset("no host-symbiont associations")
    assoc_text_start = assoc_block.start(1)
    assoc_text_end = assoc_block.start(2) if assoc_block.lastindex and assoc_block.lastindex >= 2 and assoc_block.group(2) else assoc_block.end(1)

    # extract tree Newick text for host and parasite
    def _extract_tree_text(block, is_nexus_block=False):
        body = block.group(1).strip()
        if is_nexus_block:
            # look for the TREE = ...; line inside the block
            m = re.search(r"TREE\s*[^=]*=\s*(.*?);", body, flags=re.S | re.I)
            if m:
                return m.group(1).strip(), m.start(1) + block.start(1), m.end(1) + block.start(1)
            # fallback: return entire block
            return body, block.start(1), block.end(1)
        else:
            return body, block.start(1), block.end(1)

    # detect whether we matched NEXUS style (BEGIN HOST) by looking at the original match text
    host_is_nexus = bool(re.search(r"BEGIN HOST;", host_block.group(0), flags=re.I))
    par_is_nexus = bool(re.search(r"BEGIN PARASITE;", parasite_block.group(0), flags=re.I))

    host_text, host_text_start, host_text_end = _extract_tree_text(host_block, is_nexus_block=host_is_nexus)
    parasite_text, par_text_start, par_text_end = _extract_tree_text(parasite_block, is_nexus_block=par_is_nexus)

    def _ensure_binary(tree):
        """Make every internal node have exactly two children (required by
        Capybara). Uses ete3's own built-ins rather than hand-rolled node
        surgery, which is more robust:
          - resolve_polytomy() splits any node with >2 children into a
            cascade of binary splits (zero-length branches).
          - node.delete() removes a unary (single-child) node and correctly
            reattaches its child to its parent, preserving branch lengths
            and fixing the child's `.up` pointer (unlike direct list
            assignment to `.children`, which leaves `.up` stale and can
            even drop a lone leaf child entirely).
        """
        if not tree:
            return tree

        # Split any polytomies (nodes with >2 children).
        tree.resolve_polytomy(recursive=True)

        # Collapse unary internal nodes left behind by pruning. Walk
        # bottom-up (postorder) over a snapshot of nodes so chains of
        # unary nodes fully collapse in one pass.
        for node in list(tree.traverse("postorder")):
            if node.up is not None and len(node.children) == 1:
                node.delete(preserve_branch_length=True)

        # If the root itself ended up with a single child (e.g. an entire
        # side of the tree was pruned away), promote that child to be the
        # new root instead of leaving a unary root node.
        while len(tree.children) == 1:
            tree = tree.children[0]
            tree.up = None

        return tree

    def _full_binary_issues(tree):
        """Return nodes whose child count is neither 0 nor 2."""
        if not tree:
            return ["empty tree"]
        issues = []
        for node in tree.traverse("preorder"):
            child_count = len(node.children)
            if child_count not in (0, 2):
                label = node.name or "<internal>"
                issues.append(f"{label}: {child_count} children")
        return issues

    def _assert_full_binary(tree, tree_name):
        issues = _full_binary_issues(tree)
        if issues:
            shown = ", ".join(issues[:5])
            extra = f"; {len(issues) - 5} more" if len(issues) > 5 else ""
            raise ValueError(f"{tree_name} tree is not full binary after pruning: {shown}{extra}")

    def _assert_root_has_two_children(tree, tree_name):
        while tree and len(tree.children) == 1:
            tree = tree.children[0]
            tree.up = None
        child_count = len(tree.children) if tree else 0
        if child_count != 2:
            raise ValueError(f"{tree_name} root has {child_count} children after pruning, expected 2")
        return tree

    def _assert_at_least_two_leaves(tree, tree_name):
        leaf_count = len(list(tree.iter_leaves())) if tree else 0
        if leaf_count < 2:
            raise SkipDataset(f"{tree_name} tree has {leaf_count} leaf after pruning, expected at least 2")

    def _assert_all_leaves_associated(host_tree, parasite_tree):
        host_leaf_set = {leaf.name for leaf in host_tree.iter_leaves() if leaf.name}
        parasite_leaf_set = {leaf.name for leaf in parasite_tree.iter_leaves() if leaf.name}
        active_pairs = [
            (parasite, host)
            for parasite, host in mapped_pairs
            if parasite in parasite_leaf_set and host in host_leaf_set
        ]
        associated_parasites = {parasite for parasite, host in active_pairs}
        associated_hosts = {host for parasite, host in active_pairs}
        parasites_without_host = sorted(parasite_leaf_set - associated_parasites)
        hosts_without_parasite = sorted(host_leaf_set - associated_hosts)
        if parasites_without_host or hosts_without_parasite:
            parts = []
            if parasites_without_host:
                shown = ", ".join(parasites_without_host[:5])
                extra = f"; {len(parasites_without_host) - 5} more" if len(parasites_without_host) > 5 else ""
                parts.append(f"symbiont leaves without host: {shown}{extra}")
            if hosts_without_parasite:
                shown = ", ".join(hosts_without_parasite[:5])
                extra = f"; {len(hosts_without_parasite) - 5} more" if len(hosts_without_parasite) > 5 else ""
                parts.append(f"host leaves without symbiont: {shown}{extra}")
            raise ValueError("; ".join(parts))

    def _active_association_pairs(host_tree, parasite_tree):
        host_leaf_set = {leaf.name for leaf in host_tree.iter_leaves() if leaf.name}
        parasite_leaf_set = {leaf.name for leaf in parasite_tree.iter_leaves() if leaf.name}
        return _keep_first_host_per_parasite([
            (parasite, host)
            for parasite, host in mapped_pairs
            if parasite in parasite_leaf_set and host in host_leaf_set
        ])

    def _format_distribution(active_pairs):
        lines = ["\tRANGE\n"]
        for index, (parasite, host) in enumerate(active_pairs):
            suffix = "," if index < len(active_pairs) - 1 else ""
            lines.append(f"\t\t{parasite}: {host}{suffix}\n")
        lines.append("\t;\n")
        return "".join(lines)

    def _strip_to_topology(newick_text):
        # Remove internal node labels and all branch lengths; keep leaf labels and topology.
        topology = re.sub(r"\)([^(),:;\s]+)?(?::[0-9.eE+\-]+)?", ")", newick_text)
        topology = re.sub(r"(?<=[A-Za-z0-9_.\-]):[0-9.eE+\-]+", "", topology)
        return topology.strip()

    def _parse_tree(newick_text, tree_name):
        candidates = [newick_text.strip(), _strip_to_topology(newick_text)]
        errors = []
        for candidate in candidates:
            if not candidate:
                continue
            candidate = candidate.rstrip(";") + ";"
            for tree_format in (1, 0, 3):
                try:
                    return Tree(candidate, format=tree_format)
                except Exception as exc:
                    errors.append(str(exc).split("\n")[0])
        raise ValueError(f"Could not parse {tree_name} tree after pruning preparation: {errors[-1] if errors else 'empty tree'}")

    try:
        host_tree = _parse_tree(host_text, "host")
        parasite_tree = _parse_tree(parasite_text, "symbiont")
    except Exception as exc:
        raise ValueError(str(exc)) from exc

    host_leaves = [leaf.name for leaf in host_tree.iter_leaves() if leaf.name]
    parasite_leaves = [leaf.name for leaf in parasite_tree.iter_leaves() if leaf.name]

    host_leaf_set = set(host_leaves)
    parasite_leaf_set = set(parasite_leaves)
    valid_mapped_pairs = _keep_first_host_per_parasite([
        (parasite, host)
        for parasite, host in mapped_pairs
        if parasite in parasite_leaf_set and host in host_leaf_set
    ])
    if not valid_mapped_pairs:
        raise SkipDataset("no valid leaf-to-leaf associations after removing internal-node associations")

    mapped_parasites = {p for p, h in valid_mapped_pairs}
    mapped_hosts = {h for p, h in valid_mapped_pairs}
    hosts_to_keep = [name for name in host_leaves if name in mapped_hosts]
    parasites_to_keep = [name for name in parasite_leaves if name in mapped_parasites]

    if hosts_to_keep and set(hosts_to_keep) != set(host_leaves):
        host_tree.prune(hosts_to_keep, preserve_branch_length=True)
    if parasites_to_keep and set(parasites_to_keep) != set(parasite_leaves):
        parasite_tree.prune(parasites_to_keep, preserve_branch_length=True)

    _assert_at_least_two_leaves(host_tree, "host")
    _assert_at_least_two_leaves(parasite_tree, "symbiont")
    host_tree = _ensure_binary(host_tree)
    parasite_tree = _ensure_binary(parasite_tree)
    _assert_full_binary(host_tree, "host")
    _assert_full_binary(parasite_tree, "symbiont")
    host_tree = _assert_root_has_two_children(host_tree, "host")
    parasite_tree = _assert_root_has_two_children(parasite_tree, "symbiont")
    active_pairs = _active_association_pairs(host_tree, parasite_tree)
    active_hosts = {host for parasite, host in active_pairs}
    host_leaves_after_binary = [leaf.name for leaf in host_tree.iter_leaves() if leaf.name]
    unused_host_leaves = [name for name in host_leaves_after_binary if name not in active_hosts]
    if unused_host_leaves:
        hosts_to_keep = [name for name in host_leaves_after_binary if name in active_hosts]
        if not hosts_to_keep:
            raise SkipDataset("no host leaves remain after pruning unused host leaves")
        host_tree.prune(hosts_to_keep, preserve_branch_length=True)
        _assert_at_least_two_leaves(host_tree, "host")
        host_tree = _ensure_binary(host_tree)
        _assert_full_binary(host_tree, "host")
        host_tree = _assert_root_has_two_children(host_tree, "host")
        active_pairs = _active_association_pairs(host_tree, parasite_tree)

    parasite_leaf_set = {leaf.name for leaf in parasite_tree.iter_leaves() if leaf.name}
    associated_parasites = {parasite for parasite, host in active_pairs}
    parasites_without_host = sorted(parasite_leaf_set - associated_parasites)
    if parasites_without_host:
        shown = ", ".join(parasites_without_host[:5])
        extra = f"; {len(parasites_without_host) - 5} more" if len(parasites_without_host) > 5 else ""
        raise ValueError(f"symbiont leaves without host: {shown}{extra}")

    new_host_tree = host_tree.write(format=9)
    new_parasite_tree = parasite_tree.write(format=9)
    if new_host_tree is None or new_parasite_tree is None:
        return nexus_text

    new_host_tree = new_host_tree.strip()
    if new_host_tree.endswith(";"):
        new_host_tree = new_host_tree[:-1].strip()

    new_parasite_tree = new_parasite_tree.strip()
    if new_parasite_tree.endswith(";"):
        new_parasite_tree = new_parasite_tree[:-1].strip()

    # Always emit standard Nexus. Capybara does not recognize the Cophylo
    # '# HOST_TREE' / '# PARASITE_TREE' wrapper even when its Newick text
    # is valid. Metadata from the source .tgl is not needed by Capybara.
    distribution_lines = []
    for index, (parasite, host) in enumerate(active_pairs):
        suffix = "," if index < len(active_pairs) - 1 else ""
        distribution_lines.append(f"        {parasite}: {host}{suffix}")
    distribution_text = "\n".join(distribution_lines)
    return (
        "#NEXUS\n"
        "BEGIN HOST;\n"
        f"    TREE * Host1 = {new_host_tree};\n"
        "ENDBLOCK;\n\n"
        "BEGIN PARASITE;\n"
        f"    TREE * Para1 = {new_parasite_tree};\n"
        "ENDBLOCK;\n\n"
        "BEGIN DISTRIBUTION;\n"
        "    RANGE\n"
        f"{distribution_text}\n"
        "    ;\n"
        "END;\n"
    )


def write_temp_nexus(source_path: Path, destination_path: Path):
    text = (REPO_ROOT / source_path).read_text(encoding="utf-8", errors="ignore")
    try:
        pruned_text = prune_unmapped_leaves(text)
    except SkipDataset as exc:
        raise SkipDataset(f"{source_path}: {exc}") from exc
    except ValueError as exc:
        raise ValueError(f"{source_path}: {exc}") from exc
    destination_path.write_text(pruned_text, encoding="utf-8")


In [ ]:
# Run Capybara Counter + Enumerator for each dataset and cost vector
import signal
import tempfile
import capybara.counter
import capybara.enumerator
from capybara.eucalypt.nexparser import NexusParser

event_vector_rx = re.compile(r"^\s*(?P<vector>\[(?P<counts>[0-9,\s]+)\]\s+of\s+size\s+(?P<weight>\d+))\s*$")


def extract_event_vectors(enumerator_output: str):
    vectors = []
    for line in enumerator_output.splitlines():
        match = event_vector_rx.match(line)
        if match:
            vectors.append(match.group("vector"))
    return vectors


def count_optimal_solutions(event_vectors):
    total = 0
    for event_vector in event_vectors:
        match = event_vector_rx.match(event_vector)
        if match:
            total += int(match.group("weight"))
    return total


def event_vector_frequencies(event_vectors):
    frequencies = []
    for event_vector in event_vectors:
        match = event_vector_rx.match(event_vector)
        if not match:
            continue
        counts = [int(value.strip()) for value in match.group("counts").split(",")]
        if len(counts) < 4:
            continue
        denominator = sum(counts[:4])
        size = int(match.group("weight"))
        if denominator == 0:
            cosp_frequency = 0
            switch_frequency = 0
        else:
            cosp_frequency = counts[0] / denominator
            switch_frequency = counts[2] / denominator
        frequencies.append(f"[{cosp_frequency:.6g}, {switch_frequency:.6g}] of size {size}")
    return frequencies


def closest_observed_frequency(real_cosp_freq, real_switch_freq, event_vectors):
    try:
        real_cosp_freq = float(real_cosp_freq)
        real_switch_freq = float(real_switch_freq)
        if pd.isna(real_cosp_freq) or pd.isna(real_switch_freq):
            return None, None, None
    except (TypeError, ValueError):
        return None, None, None

    closest = None
    for event_vector in event_vectors:
        match = event_vector_rx.match(event_vector)
        if not match:
            continue
        counts = [int(value.strip()) for value in match.group("counts").split(",")]
        if len(counts) < 4:
            continue
        denominator = sum(counts[:4])
        size = int(match.group("weight"))
        if denominator == 0:
            obs_cosp_freq = 0
            obs_switch_freq = 0
        else:
            obs_cosp_freq = counts[0] / denominator
            obs_switch_freq = counts[2] / denominator
        distance = math.sqrt((real_cosp_freq - obs_cosp_freq) ** 2 + (real_switch_freq - obs_switch_freq) ** 2)
        observed_frequency = f"[{obs_cosp_freq:.6g}, {obs_switch_freq:.6g}] of size {size}"
        if closest is None or distance < closest[0] or (distance == closest[0] and size > closest[2]):
            closest = (distance, observed_frequency, size)

    if closest is None:
        return None, None, None
    return closest


def _format_duration(seconds):
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    if hours:
        return f"{hours}h {minutes}m {seconds}s"
    if minutes:
        return f"{minutes}m {seconds}s"
    return f"{seconds}s"


class CapybaraTimeoutError(TimeoutError):
    pass


class capybara_timeout:
    def __init__(self, seconds):
        self.seconds = seconds
        self.previous_handler = None
        self.previous_timer = None

    def _handle_timeout(self, signum, frame):
        raise CapybaraTimeoutError(f"Capybara task exceeded {self.seconds} seconds")

    def __enter__(self):
        if self.seconds is None:
            return self
        self.previous_handler = signal.getsignal(signal.SIGALRM)
        self.previous_timer = signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, self._handle_timeout)
        signal.setitimer(signal.ITIMER_REAL, self.seconds)
        return self

    def __exit__(self, exc_type, exc, tb):
        if self.seconds is not None:
            signal.setitimer(signal.ITIMER_REAL, 0)
            signal.signal(signal.SIGALRM, self.previous_handler)
            if self.previous_timer and self.previous_timer[0] > 0:
                signal.setitimer(signal.ITIMER_REAL, *self.previous_timer)
        return False


results = []
skipped_datasets = []
capybara_setup_errors = []
valid_dataset_paths = set()
checkpoint_dir = Path("capybara_checkpoints")
checkpoint_dir.mkdir(exist_ok=True)

# Set to None to run every cost label, or set a list to run only selected costs.
# Available labels are usually: classical_1, classical_2, classical_3, transformed.
COST_LABELS_TO_RUN = ["classical_1", "classical_3", "transformed"]

# Maximum seconds allowed for one dataset/cost-vector Capybara call.
# Set to None to disable timeouts.
CAPYBARA_TIMEOUT_SECONDS = 60

capybara_tasks_df = tasks_df[tasks_df["cost_vector"].apply(lambda v: v is not None)].copy()
if COST_LABELS_TO_RUN is not None:
    capybara_tasks_df = capybara_tasks_df[capybara_tasks_df["cost_label"].isin(COST_LABELS_TO_RUN)].copy()
    available_labels = sorted(tasks_df["cost_label"].dropna().unique())
    selected_labels = sorted(capybara_tasks_df["cost_label"].dropna().unique())
    missing_labels = sorted(set(COST_LABELS_TO_RUN) - set(selected_labels))
    if missing_labels:
        raise ValueError(
            f"Requested cost labels not available after filtering: {missing_labels}. "
            f"Available labels in tasks_df: {available_labels}. "
            "Rerun the task-building cell that creates tasks_df before this Capybara run cell."
        )

candidate_dataset_paths = list(capybara_tasks_df["dataset_path"].drop_duplicates())
print("Candidate datasets before NEXUS validation:", len(candidate_dataset_paths), flush=True)
validated_nexus_dir = Path(tempfile.mkdtemp(prefix="capybara_validated_nexus_"))
validated_nexus_paths = {}

for validation_index, dataset_path in enumerate(candidate_dataset_paths, start=1):
    nex_path = validated_nexus_dir / f"dataset_{validation_index:06d}.nex"
    try:
        write_temp_nexus(dataset_path, nex_path)
        with nex_path.open() as validation_file:
            parser = NexusParser(validation_file)
            parser.read()
        valid_dataset_paths.add(dataset_path)
        validated_nexus_paths[dataset_path] = nex_path
    except SkipDataset as exc:
        skipped_datasets.append({
            "dataset_path": dataset_path,
            "skip_reason": str(exc),
        })
    except Exception as exc:
        capybara_setup_errors.append({
            "dataset_path": dataset_path,
            "error": str(exc),
        })
    finally:
        if dataset_path not in valid_dataset_paths:
            nex_path.unlink(missing_ok=True)
    if validation_index % 500 == 0 or validation_index == len(candidate_dataset_paths):
        print(
            f"Validated {validation_index}/{len(candidate_dataset_paths)} candidate datasets | "
            f"valid {len(valid_dataset_paths)} | skipped {len(skipped_datasets)} | errors {len(capybara_setup_errors)}",
            flush=True,
        )

capybara_tasks_df = capybara_tasks_df[capybara_tasks_df["dataset_path"].isin(valid_dataset_paths)].copy()

validation_skipped_df = pd.DataFrame(skipped_datasets)
validation_errors_df = pd.DataFrame(capybara_setup_errors)
validation_skipped_df.to_csv(checkpoint_dir / "validation_skipped_datasets.csv", index=False)
validation_errors_df.to_csv(checkpoint_dir / "validation_setup_errors.csv", index=False)
if validation_skipped_df.empty:
    print("Validation skipped datasets: none", flush=True)
else:
    print("Validation skipped reasons:", flush=True)
    print(validation_skipped_df["skip_reason"].value_counts().to_string(), flush=True)
if validation_errors_df.empty:
    print("Validation setup errors: none", flush=True)
else:
    print("Validation setup error types:", flush=True)
    print(validation_errors_df["error"].value_counts().head(20).to_string(), flush=True)
capybara_tasks_df = capybara_tasks_df.sort_values(["generator", "regime", "dataset_path", "cost_label"]).reset_index(drop=True)

print("Cost labels to run:", sorted(capybara_tasks_df["cost_label"].unique()), flush=True)
print("Datasets to process:", capybara_tasks_df["dataset_path"].nunique(), flush=True)
print("Task rows to process:", len(capybara_tasks_df), flush=True)

total_datasets_to_process = capybara_tasks_df["dataset_path"].nunique()
total_tasks_to_process = len(capybara_tasks_df)
task_count_by_group = capybara_tasks_df.groupby(["generator", "regime"]).size().to_dict()
task_count_by_generator = capybara_tasks_df.groupby("generator").size().to_dict()
finished_task_count_by_group = {}
finished_task_count_by_generator = {}
processed_dataset_paths = set()
completed_groups = set()
completed_generators = set()
checkpoint_every_tasks = 2000
last_checkpoint_processed_task_count = 0
processed_task_count = 0
run_start_time = time.time()
progress_every_tasks = 100


def _vec_to_str(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    if isinstance(v, Path):
        return str(v)
    if isinstance(v, (list, tuple)):
        return ", ".join(str(x) for x in v)
    return str(v)


def _events_to_str(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    if isinstance(v, (list, tuple)):
        return "; ".join(str(x) for x in v)
    return str(v)


def _safe_checkpoint_label(label):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(label)).strip("_")


def result_rows_dataframe(rows):
    # Some reconciliation counts exceed pandas/NumPy's signed 64-bit
    # integer range. Preserve those exact counts as decimal text.
    safe_rows = []
    for row in rows:
        safe_row = dict(row)
        for key, value in safe_row.items():
            if isinstance(value, int) and not -(2**63) <= value < 2**63:
                safe_row[key] = str(value)
        safe_rows.append(safe_row)
    return pd.DataFrame(safe_rows)


def checkpoint_results_df(rows):
    checkpoint_df = result_rows_dataframe(rows).copy()
    if checkpoint_df.empty:
        return checkpoint_df
    if "dataset_path" in checkpoint_df.columns:
        checkpoint_df["dataset_path"] = checkpoint_df["dataset_path"].apply(_vec_to_str)
    if "cost_vector" in checkpoint_df.columns:
        checkpoint_df["cost_vector"] = checkpoint_df["cost_vector"].apply(_vec_to_str)
    for list_col in ["event_vectors", "frequencies"]:
        if list_col in checkpoint_df.columns:
            checkpoint_df[list_col] = checkpoint_df[list_col].apply(_events_to_str)
    return checkpoint_df


def write_results_checkpoint(rows, label):
    if not rows:
        return
    checkpoint_df = checkpoint_results_df(rows)
    safe_label = _safe_checkpoint_label(label)
    checkpoint_path = checkpoint_dir / f"results_{safe_label}.csv"
    latest_path = checkpoint_dir / "results_latest.csv"
    checkpoint_df.to_csv(checkpoint_path, index=False)
    checkpoint_df.to_csv(latest_path, index=False)
    print(f"Checkpoint saved: {checkpoint_path} ({len(checkpoint_df)} rows)", flush=True)


def print_progress():
    elapsed = time.time() - run_start_time
    seconds_per_task = elapsed / processed_task_count if processed_task_count else 0
    remaining_tasks = total_tasks_to_process - processed_task_count
    eta = remaining_tasks * seconds_per_task
    completed_result_count = sum(1 for row in results if row.get("capybara_status", "completed") == "completed")
    skipped_task_count = sum(1 for row in results if row.get("capybara_status") == "skipped")
    print(
        f"{processed_task_count}/{total_tasks_to_process} tasks | "
        f"{len(processed_dataset_paths)}/{total_datasets_to_process} datasets | "
        f"completed {completed_result_count} | skipped_tasks {skipped_task_count} | "
        f"validation_skipped {len(skipped_datasets)} | errors {len(capybara_setup_errors)} | "
        f"elapsed {_format_duration(elapsed)} | ETA {_format_duration(eta)}",
        flush=True,
    )


def mark_task_rows_processed(row, task_count):
    group_key = (row["generator"], row["regime"])
    generator_key = row["generator"]
    finished_task_count_by_group[group_key] = finished_task_count_by_group.get(group_key, 0) + task_count
    finished_task_count_by_generator[generator_key] = finished_task_count_by_generator.get(generator_key, 0) + task_count

    if finished_task_count_by_group[group_key] == task_count_by_group[group_key] and group_key not in completed_groups:
        completed_groups.add(group_key)
        write_results_checkpoint(results, f"{group_key[0]}_{group_key[1]}")

    if finished_task_count_by_generator[generator_key] == task_count_by_generator[generator_key] and generator_key not in completed_generators:
        completed_generators.add(generator_key)
        write_results_checkpoint(results, generator_key)


for dataset_index, (dataset_path, dataset_tasks) in enumerate(capybara_tasks_df.groupby("dataset_path", sort=False), start=1):
    first_row = dataset_tasks.iloc[0]
    dataset_task_count = len(dataset_tasks)
    print(
        f"Starting dataset {dataset_index}/{total_datasets_to_process} | "
        f"next task {processed_task_count + 1}/{total_tasks_to_process}",
        flush=True,
    )
    nex_path = validated_nexus_paths[dataset_path]

    try:
        for cost_index, (_, row) in enumerate(dataset_tasks.iterrows(), start=1):
            cost_vector = tuple(enforce_positive_loss_cost(row["cost_vector"]))
            print(
                f"Started task {processed_task_count + 1}/{total_tasks_to_process} | "
                f"dataset {dataset_index}/{total_datasets_to_process} | "
                f"file {dataset_path} | "
                f"cost {cost_index}/{dataset_task_count} | "
                f"label {row['cost_label']} | vector {cost_vector}",
                flush=True,
            )
            out_path = Path(tempfile.mktemp(suffix=".txt"))
            task_start_time = time.time()
            try:
                with capybara_timeout(CAPYBARA_TIMEOUT_SECONDS):
                    capybara.enumerator.run(str(nex_path), str(out_path), task=2, cost_vector=cost_vector)
                enumerator_output = out_path.read_text() if out_path.exists() else ""
                event_vectors = extract_event_vectors(enumerator_output)
                event_vector_count = len(event_vectors)
                optimal_count = count_optimal_solutions(event_vectors)
                frequencies = event_vector_frequencies(event_vectors)
                min_distance, closest_frequency, closest_size = closest_observed_frequency(
                    row.get("real_cosp_freq", None), row.get("real_switch_freq", None), event_vectors
                )
                results.append({
                    "dataset_path": dataset_path,
                    "generator": row["generator"],
                    "regime": row["regime"],
                    "cost_label": row["cost_label"],
                    "real_cosp_freq": row.get("real_cosp_freq", None),
                    "cosp_freq": row.get("cosp_freq", None),
                    "dup_freq": row.get("dup_freq", None),
                    "real_switch_freq": row.get("real_switch_freq", None),
                    "switch_freq": row.get("switch_freq", None),
                    "loss_freq": row.get("loss_freq", None),
                    "cost_vector": row["cost_vector"],
                    "event_vector_count": event_vector_count,
                    "optimal_count": optimal_count,
                    "event_vectors": event_vectors,
                    "frequencies": frequencies,
                    "min_real_observed_distance": min_distance,
                    "closest_observed_frequency": closest_frequency,
                    "closest_observed_size": closest_size,
                    "task_elapsed_seconds": time.time() - task_start_time,
                    "capybara_status": "completed",
                    "skip_reason": "",
                })
            except CapybaraTimeoutError as exc:
                elapsed_seconds = time.time() - task_start_time
                capybara_setup_errors.append({
                    "dataset_path": dataset_path,
                    "cost_label": row["cost_label"],
                    "cost_vector": cost_vector,
                    "error_type": "timeout",
                    "error": str(exc),
                    "task_elapsed_seconds": elapsed_seconds,
                })
                results.append({
                    "dataset_path": dataset_path,
                    "generator": row["generator"],
                    "regime": row["regime"],
                    "cost_label": row["cost_label"],
                    "real_cosp_freq": row.get("real_cosp_freq", None),
                    "cosp_freq": row.get("cosp_freq", None),
                    "dup_freq": row.get("dup_freq", None),
                    "real_switch_freq": row.get("real_switch_freq", None),
                    "switch_freq": row.get("switch_freq", None),
                    "loss_freq": row.get("loss_freq", None),
                    "cost_vector": row["cost_vector"],
                    "event_vector_count": 0,
                    "optimal_count": None,
                    "event_vectors": [],
                    "frequencies": [],
                    "min_real_observed_distance": None,
                    "closest_observed_frequency": None,
                    "closest_observed_size": None,
                    "task_elapsed_seconds": elapsed_seconds,
                    "capybara_status": "skipped",
                    "skip_reason": str(exc),
                })
                print(
                    f"Timed out task {processed_task_count + 1}/{total_tasks_to_process} after {_format_duration(elapsed_seconds)} | "
                    f"file {dataset_path} | label {row['cost_label']} | vector {cost_vector}",
                    flush=True,
                )
            except Exception as exc:
                elapsed_seconds = time.time() - task_start_time
                capybara_setup_errors.append({
                    "dataset_path": dataset_path,
                    "cost_label": row["cost_label"],
                    "cost_vector": cost_vector,
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                    "task_elapsed_seconds": elapsed_seconds,
                })
                results.append({
                    "dataset_path": dataset_path,
                    "generator": row["generator"],
                    "regime": row["regime"],
                    "cost_label": row["cost_label"],
                    "real_cosp_freq": row.get("real_cosp_freq", None),
                    "cosp_freq": row.get("cosp_freq", None),
                    "dup_freq": row.get("dup_freq", None),
                    "real_switch_freq": row.get("real_switch_freq", None),
                    "switch_freq": row.get("switch_freq", None),
                    "loss_freq": row.get("loss_freq", None),
                    "cost_vector": row["cost_vector"],
                    "event_vector_count": 0,
                    "optimal_count": None,
                    "event_vectors": [],
                    "frequencies": [],
                    "min_real_observed_distance": None,
                    "closest_observed_frequency": None,
                    "closest_observed_size": None,
                    "task_elapsed_seconds": elapsed_seconds,
                    "capybara_status": "skipped",
                    "skip_reason": f"{type(exc).__name__}: {exc}",
                })
                print(
                    f"Failed task {processed_task_count + 1}/{total_tasks_to_process} after {_format_duration(elapsed_seconds)} | "
                    f"file {dataset_path} | label {row['cost_label']} | vector {cost_vector} | error {exc}",
                    flush=True,
                )
            finally:
                out_path.unlink(missing_ok=True)

            processed_task_count += 1
            mark_task_rows_processed(row, 1)
            if processed_task_count % progress_every_tasks == 0 or processed_task_count == total_tasks_to_process:
                print_progress()
            if processed_task_count - last_checkpoint_processed_task_count >= checkpoint_every_tasks:
                write_results_checkpoint(results, f"task_{processed_task_count:06d}")
                last_checkpoint_processed_task_count = processed_task_count
    finally:
        nex_path.unlink(missing_ok=True)

    processed_dataset_paths.add(dataset_path)

write_results_checkpoint(results, "final")
results_df = result_rows_dataframe(results)
skipped_df = pd.DataFrame(skipped_datasets)
capybara_setup_errors_df = pd.DataFrame(capybara_setup_errors)

def attach_generator_regime(df):
    df = df.copy()
    if df.empty or "dataset_path" not in df.columns:
        return df
    metadata = df["dataset_path"].apply(synthetic_metadata)
    df["generator"] = metadata.str[0]
    df["regime"] = metadata.str[1]
    return df

skipped_df = attach_generator_regime(skipped_df)
capybara_setup_errors_df = attach_generator_regime(capybara_setup_errors_df)

if skipped_df.empty:
    skipped_by_generator_df = pd.DataFrame(columns=["generator", "skipped_datasets"])
    print("Skipped datasets by generator: none", flush=True)
else:
    skipped_by_generator_df = (
        skipped_df.groupby("generator", dropna=False)
        .size()
        .reset_index(name="skipped_datasets")
        .sort_values("generator")
    )
    print("Skipped datasets by generator:", flush=True)
    print(skipped_by_generator_df.to_string(index=False), flush=True)

if capybara_setup_errors_df.empty:
    errors_by_generator_df = pd.DataFrame(columns=["generator", "setup_errors"])
    print("Setup errors by generator: none", flush=True)
else:
    errors_by_generator_df = (
        capybara_setup_errors_df.groupby("generator", dropna=False)
        .size()
        .reset_index(name="setup_errors")
        .sort_values("generator")
    )
    print("Setup errors by generator:", flush=True)
    print(errors_by_generator_df.to_string(index=False), flush=True)

skipped_df.to_csv(checkpoint_dir / "skipped_datasets.csv", index=False)
capybara_setup_errors_df.to_csv(checkpoint_dir / "setup_errors.csv", index=False)
skipped_by_generator_df.to_csv(checkpoint_dir / "skipped_by_generator.csv", index=False)
errors_by_generator_df.to_csv(checkpoint_dir / "setup_errors_by_generator.csv", index=False)
results_df.head()


## Export Results to Excel
Write one sheet per generator + regime combination, producing 12 sheets total in the workbook.

In [ ]:
!pip install openpyxl


In [ ]:
output_file = Path("synthetic_data_capybara_results.xlsx")
EXCEL_CELL_TEXT_LIMIT = 32767
EXCEL_MAX_ROWS = 1048576

def excel_safe_value(value):
    """Convert a result to a value that openpyxl can safely store."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    if isinstance(value, (list, tuple, dict, set, Path)):
        value = str(value)
    # Excel/openpyxl cannot safely store arbitrarily large Python integers.
    if isinstance(value, (int, np.integer)) and not -(2**63) <= value < 2**63:
        value = str(value)
    if isinstance(value, str):
        # Remove control characters forbidden by the XLSX specification.
        value = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", "", value)
        if len(value) > EXCEL_CELL_TEXT_LIMIT:
            value = value[:EXCEL_CELL_TEXT_LIMIT - 16] + " ...[truncated]"
    return value

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    # Write a visible sheet immediately so openpyxl can always close cleanly.
    pd.DataFrame([{"note": "Capybara results export"}]).to_excel(
        writer, sheet_name="export_info", index=False
    )
    # Ensure we always write at least one visible sheet
    if 'results_df' not in globals() or results_df.empty:
        empty_df = pd.DataFrame([{"note": "No Capybara results available (no runs or capybara missing)"}])
        empty_df.to_excel(writer, sheet_name="no_results", index=False)
    else:
        for (generator, regime), group in results_df.groupby(["generator", "regime"]):
            sheet_name = f"{generator}_{regime}"[:31]
            group_to_write = group.copy()

            def vec_to_str(v):
                if v is None or (isinstance(v, float) and pd.isna(v)):
                    return ""
                if isinstance(v, (list, tuple)):
                    return ", ".join(str(x) for x in v)
                return str(v)

            def events_to_str(v):
                if v is None or (isinstance(v, float) and pd.isna(v)):
                    return ""
                if isinstance(v, (list, tuple)):
                    return "; ".join(str(x) for x in v)
                return str(v)

            # write the existing frequency columns that are still present
            for freq_col in ["real_cosp_freq", "cosp_freq", "dup_freq", "real_switch_freq", "switch_freq", "loss_freq"]:
                if freq_col in group_to_write.columns:
                    group_to_write[freq_col] = group_to_write[freq_col].apply(vec_to_str)
            if "cost_vector" in group_to_write.columns:
                group_to_write["cost_vector"] = group_to_write["cost_vector"].apply(vec_to_str)
            for list_col in ["event_vectors", "frequencies"]:
                if list_col in group_to_write.columns:
                    group_to_write[list_col] = group_to_write[list_col].apply(events_to_str)
            # Sanitize column-by-column to avoid pandas trying to coerce huge integers.
            # This changes only the export copy; results_df remains unchanged.
            for column in group_to_write.columns:
                group_to_write[column] = pd.Series(
                    [excel_safe_value(value) for value in group_to_write[column]],
                    index=group_to_write.index,
                    dtype=object,
                )

            # Split a group only if it exceeds Excel's worksheet row limit.
            rows_per_sheet = EXCEL_MAX_ROWS - 1  # reserve one row for headers
            for part_number, start in enumerate(range(0, len(group_to_write), rows_per_sheet), start=1):
                part = group_to_write.iloc[start:start + rows_per_sheet]
                part_sheet_name = sheet_name if part_number == 1 else f"{sheet_name[:27]}_{part_number}"
                part.to_excel(writer, sheet_name=part_sheet_name[:31], index=False)

print("Excel file saved to:", output_file)